Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('../results', exist_ok=True)

Load & Gabungkan Ulang Dataset

In [2]:
orders   = pd.read_csv('../data/olist_orders_dataset.csv')
items    = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
transl   = pd.read_csv('../data/product_category_name_translation.csv')

df = items.merge(orders, on='order_id') \
          .merge(products, on='product_id') \
          .merge(transl, on='product_category_name', how='left')

print("Shape awal:", df.shape)

Shape awal: (112650, 23)


 Filter Order yang Valid

In [3]:
# Hanya ambil order yang sudah delivered
df = df[df['order_status'] == 'delivered']

# Hapus baris dengan tanggal kosong
df = df.dropna(subset=['order_purchase_timestamp'])

print("Shape setelah filter:", df.shape)

Shape setelah filter: (110197, 23)


Konversi Tanggal & Fitur Waktu

In [4]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

df['tahun']       = df['order_purchase_timestamp'].dt.year
df['bulan']       = df['order_purchase_timestamp'].dt.month
df['minggu']      = df['order_purchase_timestamp'].dt.isocalendar().week.astype(int)
df['hari_dalam_minggu'] = df['order_purchase_timestamp'].dt.dayofweek
df['kuartal']     = df['order_purchase_timestamp'].dt.quarter

print(df[['tahun','bulan','minggu','hari_dalam_minggu','kuartal']].head())

   tahun  bulan  minggu  hari_dalam_minggu  kuartal
0   2017      9      37                  2        3
1   2017      4      17                  2        2
2   2018      1       2                  6        1
3   2018      8      32                  2        3
4   2017      2       5                  5        1


Agregasi Target: Jumlah Order per Minggu per Kategori

In [5]:
# Ini yang jadi target prediksi kita
target = df.groupby(['tahun', 'bulan', 'minggu', 
                     'product_category_name_english'])['order_id'] \
            .count().reset_index()

target.columns = ['tahun', 'bulan', 'minggu', 'kategori', 'jumlah_order']

print("Shape target:", target.shape)
print(target.head(10))

Shape target: (4931, 5)
   tahun  bulan  minggu               kategori  jumlah_order
0   2016      9      37          health_beauty             3
1   2016     10      40       air_conditioning             8
2   2016     10      40                  audio             1
3   2016     10      40                   auto             7
4   2016     10      40                   baby            11
5   2016     10      40         bed_bath_table             8
6   2016     10      40  computers_accessories            11
7   2016     10      40         consoles_games             5
8   2016     10      40             cool_stuff             7
9   2016     10      40    diapers_and_hygiene             1


Encoding Kategori & Simpan Dataset Bersih

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
target['kategori_encoded'] = le.fit_transform(target['kategori'])

# Simpan mapping kategori biar bisa dipakai nanti
mapping = pd.DataFrame({
    'kategori': le.classes_,
    'encoded': range(len(le.classes_))
})
mapping.to_csv('../data/mapping_kategori.csv', index=False)

# Simpan dataset bersih
target.to_csv('../data/dataset_bersih.csv', index=False)

print("Dataset bersih tersimpan!")
print("Shape final:", target.shape)
target.head()

Dataset bersih tersimpan!
Shape final: (4931, 6)


,tahun,bulan,minggu,kategori,jumlah_order,kategori_encoded
0,2016,9,37,health_beauty,3,43
1,2016,10,40,air_conditioning,8,1
2,2016,10,40,audio,1,4
3,2016,10,40,auto,7,5
4,2016,10,40,baby,11,6
